# DermaVision AI: train the skin lesion model (Google Colab)
Run the cells **in order**. Before you start: **Runtime → Change runtime type → T4 GPU**.
You need: the project zip (`DermaVisionAI.zip`) and a free Kaggle account with an API token (`kaggle.json`).
Educational use only: HAM10000 is licensed CC BY-NC 4.0.

## 1. Check the GPU

In [ ]:
!nvidia-smi | head -n 12

## 2. Upload and unzip the project
When prompted, choose `DermaVisionAI.zip`.

In [ ]:
from google.colab import files
files.upload()
!unzip -q -o DermaVisionAI.zip
%cd DermaVisionAI/ml
!pip install -q pandas scikit-learn matplotlib

## 3. Download HAM10000 from Kaggle
Kaggle → Settings → API → *Create New Token* downloads `kaggle.json`. Upload it when prompted.

In [ ]:
from google.colab import files
files.upload()   # choose kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data/HAM10000 --unzip
!ls data/HAM10000

## 4. Split the data by lesion (prevents train/test leakage)

In [ ]:
!python prepare_data.py --data-dir data/HAM10000

## 5. Smoke test (about 1 minute)
Confirms everything works before the long run. Expect low scores here.

In [ ]:
!python train.py --data-dir data/HAM10000/splits --epochs 1 --warmup-epochs 1 --max-samples 256 --workers 2 --out /tmp/smoke.pt

## 6. Full training (about 30 to 60 minutes on a T4)

In [ ]:
!python train.py --data-dir data/HAM10000/splits --epochs 25 --workers 2

## 7. Evaluate on the untouched test split
Look at **recall for `mel`** (melanoma), not just accuracy.

In [ ]:
!python evaluate.py --data-dir data/HAM10000/splits --workers 2
from IPython.display import Image
Image('../backend/weights/confusion_matrix.png')

## 8. Download the trained weights
Put the file in `backend/weights/` of your project, set `DEMO_MODE=false`, and restart the API.

In [ ]:
from google.colab import files
files.download('../backend/weights/dermavision_effnet_b0.pt')